# 04 — Shootout: Qwen3-1.7B con el pipeline del grafo

**Objetivo:** medir si un modelo ~2.2× más ligero (Qwen3-1.7B, GGUF Q4_K_M de 1.1 GB) mantiene
la calidad del pipeline v3 (grafo de dominio + reglas duras) bajando el tiempo por anuncio.

**Hipótesis:** el grafo redujo el trabajo cognitivo del modelo — ya no inventa ángulos, solo
*redacta* a partir de hechos curados — así que un 1.7B debería rendir casi igual que el 4B en
esta tarea, a la mitad del tiempo.

**⚠️ Particularidad de Qwen3-1.7B:** a diferencia del 4B-Instruct-2507, este es un modelo
*híbrido con razonamiento*: por default "piensa" (bloque `<think>`) antes de responder, lo que
destruiría la ganancia de velocidad. Se desactiva con la directiva `/no_think` al final del
system prompt.

**Prerrequisitos:** haber ejecutado el notebook 03 (genera `grafo_dominio.json` y, si corriste
el A/B completo, `resultados_ab_grafo.csv` para comparar contra el 4B).

**Criterio de decisión:** el 1.7B gana si pasa 7/7 verificaciones automáticas (sin precio, sin
contacto) y en la lectura manual no se siente "robótico" frente al 4B. Si empata en calidad,
el tiempo decide.

## 1. Descarga y carga del modelo

Mismo patrón que el notebook 01: descarga idempotente del GGUF Q4_K_M al cache local
`models_registry/llm/` (repo público de unsloth, no requiere token).

In [1]:
import json
import re
import time
import unicodedata
from datetime import date
from pathlib import Path

import networkx as nx
import pandas as pd
import psutil
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

MODELS_DIR = Path("../../models_registry/llm").resolve()

model_path = hf_hub_download(
    repo_id="unsloth/Qwen3-1.7B-GGUF",
    filename="Qwen3-1.7B-Q4_K_M.gguf",  # ~1.1 GB
    local_dir=MODELS_DIR,
)
print(f"Modelo en: {model_path} ({Path(model_path).stat().st_size / 1e9:.2f} GB)")

proceso = psutil.Process()
ram_antes = proceso.memory_info().rss / 1e9
llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_threads=4,
    verbose=False,
)
print(f"RAM del modelo: +{proceso.memory_info().rss / 1e9 - ram_antes:.2f} GB")

with open("drafts_ejemplo.json") as f:
    DRAFTS = [d["draft"] for d in json.load(f)]

AÑO_ACTUAL = date.today().year
print(f"{len(DRAFTS)} drafts de ejemplo | año actual: {AÑO_ACTUAL}")

Modelo en: /home/aleosh/Documentos/Ingeniería en Software/9no Cuatrimestre/Integrador/vps/vivia-ai/models_registry/llm/Qwen3-1.7B-Q4_K_M.gguf (1.11 GB)
RAM del modelo: +2.30 GB
7 drafts de ejemplo | año actual: 2026


## 2. Carga del grafo de dominio

Aquí NO se reconstruye el grafo: se carga el artefacto `grafo_dominio.json` que serializó el
notebook 03 — el mismo flujo que tendrá el worker en producción (GGUF + grafo como artefactos).
Si editas el grafo, hazlo en la sección 2 del notebook 03 y re-ejecuta su serialización.

In [2]:
with open("grafo_dominio.json") as f:
    G = nx.node_link_graph(json.load(f), edges="edges", directed=True)

print(f"Grafo cargado: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")


def _normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower().strip())
    return "".join(c for c in texto if not unicodedata.combining(c))

ALIAS_A_AMENIDAD = {
    _normalizar(alias): nodo
    for nodo, data in G.nodes(data=True) if data["tipo"] == "amenidad"
    for alias in data["aliases"] + [data["nombre"]]
}

Grafo cargado: 33 nodos, 26 aristas


## 3. Pipeline v3 (idéntico al notebook 03)

Parser del draft real (sin precio, sin calle/números) y consulta del grafo. Se mantiene igual
para que la única variable del experimento sea el modelo.

In [3]:
def draft_json_a_texto(draft: dict) -> str:
    operacion = "RENTA" if draft["availableToRent"] else "VENTA"
    antiguedad = AÑO_ACTUAL - draft["constructionYear"]
    lineas = [
        f"Tipo de propiedad: {draft['propertyType']['name']}",
        f"Operación: {operacion}",
        f"Colonia: {draft['address']['neighborhoodName']}",
        f"Superficie: {draft['areaM2']:.0f} m²",
        f"Recámaras: {draft['bedrooms']}",
        f"Baños: {draft['bathrooms']}",
    ]
    if draft["parkingSpaces"] > 0:
        lineas.append(f"Estacionamientos: {draft['parkingSpaces']}")
    lineas.append(
        f"Año de construcción: {draft['constructionYear']}"
        + (f" ({antiguedad} años de antigüedad)" if antiguedad > 1 else " (a estrenar)")
    )
    lineas.append(f"En condominio: {'sí' if draft['condominium'] else 'no'}")
    if draft["amenities"]:
        lineas.append(f"Amenidades: {', '.join(draft['amenities'])}")
    return "\n".join(lineas)


def audiencias_para(draft: dict) -> list[str]:
    tipo = draft["propertyType"]["name"]
    rec, renta = draft["bedrooms"], draft["availableToRent"]
    if rec >= 4 or (rec >= 3 and tipo == "CASA"):
        return ["audiencia:familia_grande"]
    if rec >= 2:
        return ["audiencia:familia_pareja"]
    if renta:
        return ["audiencia:profesionista", "audiencia:persona_sola"]
    return ["audiencia:persona_sola"]


def contexto_desde_grafo(draft: dict) -> str:
    lineas = ["HECHOS Y ÁNGULOS APROBADOS (única fuente permitida además del draft):"]
    temas_usados: set[str] = set()

    op = "RENTA" if draft["availableToRent"] else "VENTA"
    lineas.append(f"- Operación {op}: {G.nodes[f'operacion:{op}']['tono']}.")
    nodo_tipo = f"tipo:{draft['propertyType']['name']}"
    if nodo_tipo in G:
        lineas.append(f"- Un(a) {draft['propertyType']['name'].lower()} {G.nodes[nodo_tipo]['narrativa']}.")
        temas_usados.update(G.successors(nodo_tipo))

    desconocidas = []
    for amenidad in draft["amenities"]:
        nodo = ALIAS_A_AMENIDAD.get(_normalizar(amenidad))
        if nodo is None:
            desconocidas.append(amenidad)
            continue
        for tema in G.successors(nodo):
            if tema in temas_usados:
                continue
            temas_usados.add(tema)
            frase = G.nodes[tema]["frases"][0]
            lineas.append(f"- {G.nodes[nodo]['nombre']} → {G.nodes[tema]['nombre']}: \"{frase}\".")
    if desconocidas:
        lineas.append(f"- Amenidades sin ángulo aprobado (menciónalas SOLO por su nombre): {', '.join(desconocidas)}.")

    frases_aud = [G.nodes[a]["frase"] for a in audiencias_para(draft)]
    lineas.append(f"- Audiencia sugerida: {'; '.join(frases_aud)}.")

    antiguedad = AÑO_ACTUAL - draft["constructionYear"]
    if antiguedad <= 1:
        lineas.append("- Antigüedad: a estrenar — puedes destacar que es de reciente construcción.")
    elif antiguedad <= 10:
        lineas.append("- Antigüedad: construcción reciente.")
    elif antiguedad > 30:
        lineas.append("- Antigüedad: más de 30 años — nómbrala como carácter e historia; "
                      "NO afirmes remodelaciones ni buen estado que el draft no indica.")
    if draft["areaM2"] >= 180:
        lineas.append("- Superficie: puedes describirla como amplia.")
    elif draft["areaM2"] <= 50:
        lineas.append("- Superficie: descríbela como compacta y eficiente, no como amplia.")
    return "\n".join(lineas)


print(contexto_desde_grafo(DRAFTS[0]))

HECHOS Y ÁNGULOS APROBADOS (única fuente permitida además del draft):
- Operación RENTA: hogar disponible para renta; enfatiza la vida diaria y la facilidad de instalarse; jamás menciones montos, rentas ni mensualidades.
- Un(a) casa se narra por sus espacios, su independencia y la vida cotidiana que permite.
- terraza → vida al aire libre: "espacios para disfrutar el aire libre sin salir de casa".
- gimnasio → bienestar: "facilita una rutina activa y saludable".
- Audiencia sugerida: familias que necesitan espacio para todos.
- Antigüedad: a estrenar — puedes destacar que es de reciente construcción.
- Superficie: puedes describirla como amplia.


## 4. Prompt v3 con `/no_think`

El mismo system prompt v3 del notebook 03, con la directiva `/no_think` al final para
desactivar el razonamiento del modelo híbrido. (La gramática JSON además le impide emitir el
bloque `<think>`, pero la directiva alinea al modelo con esa restricción en vez de pelear
contra ella.)

In [4]:
SYSTEM_PROMPT_V3 = """Eres un redactor inmobiliario profesional de México. Recibirás el DRAFT de una \
propiedad y un bloque de HECHOS Y ÁNGULOS APROBADOS. Escribirás el anuncio para un portal inmobiliario.

Reglas estrictas:
1. Usa ÚNICAMENTE la información del DRAFT y de los HECHOS Y ÁNGULOS APROBADOS. Puedes \
parafrasear las frases aprobadas con naturalidad, pero NO agregues características, lugares, \
vistas, cercanías ni cualidades que no aparezcan ahí.
2. PROHIBIDO mencionar precios, montos, rentas, mensualidades o cualquier cifra monetaria.
3. PROHIBIDO incluir datos de contacto, invitaciones a llamar, escribir o agendar visitas, \
y lenguaje de urgencia ("aprovecha", "no te lo pierdas", "últimos días").
4. La única ubicación que puedes mencionar es el nombre de la colonia, tal como aparece en el draft.
5. Usa español de México y vocabulario de la región: "recámaras" (nunca "habitaciones") y \
"estacionamientos" (nunca "cajones de estacionamiento" ni "plazas de garaje"). Si el draft no \
menciona estacionamientos, no hables de ellos.
6. El título debe tener máximo 10 palabras, atractivo sin ser sensacionalista.
7. La descripción debe tener entre 21 y 70 palabras, en párrafos fluidos (sin listas), \
con tono cálido y humano, sin mayúsculas sostenidas ni signos de admiración excesivos.

Responde exclusivamente con un JSON con las claves "titulo" y "descripcion". /no_think"""

ESQUEMA_ANUNCIO = {
    "type": "object",
    "properties": {"titulo": {"type": "string"}, "descripcion": {"type": "string"}},
    "required": ["titulo", "descripcion"],
}


def generar_v3(draft: dict, temperature: float = 1.0, max_tokens: int = 512) -> dict:
    mensaje = (f"DRAFT DE LA PROPIEDAD:\n{draft_json_a_texto(draft)}\n\n"
               f"{contexto_desde_grafo(draft)}")
    t0 = time.perf_counter()
    salida = llm.create_chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_V3},
            {"role": "user", "content": mensaje},
        ],
        response_format={"type": "json_object", "schema": ESQUEMA_ANUNCIO},
        temperature=temperature,
        max_tokens=max_tokens,
    )
    anuncio = json.loads(salida["choices"][0]["message"]["content"])
    anuncio["tiempo_s"] = round(time.perf_counter() - t0, 1)
    anuncio["tokens_prompt"] = salida["usage"]["prompt_tokens"]
    return anuncio


# Demo con el draft real del ejemplo
demo = generar_v3(DRAFTS[0])
print(f"⏱ {demo['tiempo_s']} s | prompt: {demo['tokens_prompt']} tokens\n")
print(f"TÍTULO: {demo['titulo']}\n\n{demo['descripcion']}")

⏱ 19.8 s | prompt: 703 tokens

TÍTULO: Casa de 200 m² en Prudencio Moscoso

Esta casa de 200 m² ofrece espacio para una familia completa, con 4 recámaras y 3 baños. La terraza es ideal para disfrutar del aire libre, mientras que el gimnasio fomenta una rutina activa y saludable. Con una construcción a estrenar, es una oportunidad única para vivir en un hogar moderno y funcional.


## 5. Corrida sobre los 7 drafts + verificación automática

In [5]:
RE_PRECIO = re.compile(r"\$|\bprecio\b|\bmxn\b|\bpesos?\b|mensualidad|mill[oó]n|\bmonto\b", re.I)
RE_CONTACTO = re.compile(
    r"cont[aá]ct|ll[aá]m[ae]|tel[eé]fono|whatsapp|escr[ií]b[ae]|agend[ae]|vis[ií]t[ae]|"
    r"aprovecha|no te lo pierdas|[uú]ltimos d[ií]as|cita", re.I)

filas = []
for draft in DRAFTS:
    anuncio = generar_v3(draft)
    texto = f"{anuncio['titulo']} {anuncio['descripcion']}"
    checks = {
        "menciona_precio": bool(RE_PRECIO.search(texto)),
        "menciona_contacto": bool(RE_CONTACTO.search(texto)),
        "n_palabras": len(anuncio["descripcion"].split()),
    }
    filas.append({"draft_id": draft["id"], "variante": "v3-1.7B",
                  "titulo": anuncio["titulo"], "descripcion": anuncio["descripcion"],
                  "tokens_prompt": anuncio["tokens_prompt"], "tiempo_s": anuncio["tiempo_s"],
                  **checks})
    marca = "⚠️" if checks["menciona_precio"] or checks["menciona_contacto"] else "✓"
    print(f"{marca} {draft['id']}: {anuncio['titulo']} ({anuncio['tiempo_s']} s)")

df_17b = pd.DataFrame(filas)
df_17b.to_csv("resultados_qwen17b.csv", index=False)

print("\n--- Resumen Qwen3-1.7B ---")
print(f"Violaciones precio/contacto: {int(df_17b['menciona_precio'].sum() + df_17b['menciona_contacto'].sum())}/7")
print(f"Tiempo promedio: {df_17b['tiempo_s'].mean():.1f} s | palabras promedio: {df_17b['n_palabras'].mean():.0f}")

✓ draft-001: Casa de 200 m² en Prudencio Moscoso (12.6 s)
✓ draft-002: Departamento en El Cerrillo (12.0 s)
✓ draft-003: Casa en Barrio de Guadalupe (14.6 s)
✓ draft-004: Departamento en La Isla (12.1 s)
✓ draft-005: Casa 31 de Marzo (13.2 s)
✓ draft-006: Departamento en Real del Monte (13.9 s)
✓ draft-007: Loft moderno en Centro (11.6 s)

--- Resumen Qwen3-1.7B ---
Violaciones precio/contacto: 0/7
Tiempo promedio: 12.9 s | palabras promedio: 64


## 6. Comparativa contra el 4B (resultados del notebook 03)

In [ ]:
if Path("resultados_ab_grafo.csv").exists():
    df_4b = pd.read_csv("resultados_ab_grafo.csv")
    df_4b_v3 = df_4b[df_4b["variante"] == "v3"].assign(variante="v3-4B")
    df_comp = pd.concat([df_4b_v3, df_17b], ignore_index=True)

    print("--- Resumen por modelo ---")
    display(df_comp.groupby("variante")[["menciona_precio", "menciona_contacto",
                                          "tiempo_s", "n_palabras"]].agg(
        {"menciona_precio": "sum", "menciona_contacto": "sum",
         "tiempo_s": "mean", "n_palabras": "mean"}).round(1))

    # Lectura lado a lado del draft del ejemplo real
    for _, fila in df_comp[df_comp["draft_id"] == "draft-001"].iterrows():
        print(f"\n{'=' * 70}\n### {fila['variante']} — {fila['titulo']} ({fila['tiempo_s']} s)\n")
        print(fila["descripcion"])
else:
    print("No existe resultados_ab_grafo.csv — corre el A/B del notebook 03 para comparar.")

--- Resumen por modelo ---


,menciona_precio,menciona_contacto,tiempo_s,n_palabras
variante,,,,
v3-1.7B,0,0,12.9,63.9
v3-4B,0,0,26.1,59.7



### v3-4B — Casa de reciente construcción en Prudencio Moscoso (35.7 s)

Una casa amplia y moderna, a estrenar, en el corazón de Prudencio Moscoso. Ofrece espacio para vivir con tranquilidad y bienestar. Cuenta con terraza para disfrutar el aire libre sin salir de casa y un gimnasio para mantener una rutina activa y saludable. Ideal para familias que buscan un hogar con independencia y espacios para todos.

### v3-1.7B — Casa de 200 m² en Prudencio Moscoso (12.6 s)

La casa de 200 m² en Prudencio Moscoso ofrece espacio amplio para vivir en un ambiente acogedor. Con 4 recámaras, 3 baños y 2 estacionamientos, es ideal para familias que buscan independencia y comodidad. La terraza y el gimnasio permiten disfrutar de la vida al aire libre y mantener una rutina activa. A estrenar, su diseño moderno combina funcionalidad y estilo, garantizando una experiencia de vida sana y acogedora.


: 

## 7. Evaluación

- [ ] **Verificaciones automáticas:** 0 violaciones de precio/contacto en 7/7
- [ ] **Fidelidad:** solo hechos del draft + ángulos del grafo
- [ ] **Humanización:** en la lectura lado a lado, ¿el 1.7B se siente "robótico" o repetitivo
      frente al 4B? ¿Parafrasea las frases del grafo o las copia textuales?
- [ ] **Español MX:** natural, sin calcos raros (los modelos chicos degradan primero aquí)
- [ ] **Velocidad:** tiempo promedio vs el 4B (esperado: ~2× más rápido)

### Decisión

| Resultado | Acción |
|---|---|
| Calidad comparable | Adoptar 1.7B: ~1.2 GB en RAM y ~2× velocidad en el VPS |
| Calidad notablemente peor | Quedarse con el 4B; explorar *speculative decoding* en `llama-server` (borrador 0.6B) |
| Intermedio | Probar LFM2-2.6B (1.56 GB) como punto medio |

El modelo elegido aquí es el que se lleva al benchmark del VPS y al diseño del worker.